# Notebook 02: Social Media Post Effectiveness (Explanatory)

## Section 1 — Problem Framing

**Business Problem:** Identify which social media post factors drive donation referrals so the social media team can optimize their posting strategy.

**Approach:** Explanatory OLS regression — we interpret coefficients to understand *why* certain posts generate more donation referrals, not to maximize prediction accuracy.

**Target Variable:** `donation_referrals` (continuous count of donation referrals per post)

**Key Design Decision:** We use ONLY controllable post factors as features (platform, post type, media type, etc.). Engagement metrics like impressions, reach, likes, shares, and comments are *outcomes* of the same process, not causes of donations. Including them would introduce endogeneity bias.

**Stakeholders:** Social media team, fundraising staff

## Section 2 — Data Acquisition and Preparation

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [ ]:
# Load data
from sqlalchemy import create_engine
import os

DB_HOST = os.environ.get("DB_HOST", "localhost")
DB_PORT = os.environ.get("DB_PORT", "5432")
DB_NAME = os.environ.get("DB_NAME", "harbor_of_hope")
DB_USER = os.environ.get("DB_USER", "waylansmac")
DB_PASS = os.environ.get("DB_PASS", "")
CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(CONNECTION_STRING)

smp = pd.read_sql("SELECT * FROM social_media_posts", engine)
print(f"Dataset shape: {smp.shape}")
print(f"\nColumn names:\n{list(smp.columns)}")
print(f"\nTarget variable (donation_referrals) stats:")
print(smp['donation_referrals'].describe())

In [3]:
# Feature selection — ONLY controllable post factors
# DO NOT include engagement outcomes: impressions, reach, likes, shares, comments, saves, click_throughs, video_views, engagement_rate, profile_visits
categorical_features = ['platform', 'post_type', 'media_type', 'day_of_week',
                        'content_topic', 'sentiment_tone']
binary_features = ['has_call_to_action', 'features_resident_story', 'is_boosted']
numeric_features = ['num_hashtags', 'caption_length', 'post_hour', 'follower_count_at_post']

# Convert binary features to int
for col in binary_features:
    smp[col] = smp[col].astype(int)

# One-hot encode categoricals
features_df = pd.get_dummies(smp[categorical_features + binary_features + numeric_features],
                              columns=categorical_features, drop_first=True, dtype=int)

y = smp['donation_referrals']

# Add constant for statsmodels OLS
X_full = sm.add_constant(features_df)

print(f"Feature matrix shape: {X_full.shape}")
print(f"\nFeatures used ({X_full.shape[1]-1} total, plus constant):")
for col in X_full.columns[1:]:
    print(f"  - {col}")

Feature matrix shape: (812, 42)

Features used (41 total, plus constant):
  - has_call_to_action
  - features_resident_story
  - is_boosted
  - num_hashtags
  - caption_length
  - post_hour
  - follower_count_at_post
  - platform_Instagram
  - platform_LinkedIn
  - platform_TikTok
  - platform_Twitter
  - platform_WhatsApp
  - platform_YouTube
  - post_type_EducationalContent
  - post_type_EventPromotion
  - post_type_FundraisingAppeal
  - post_type_ImpactStory
  - post_type_ThankYou
  - media_type_Photo
  - media_type_Reel
  - media_type_Text
  - media_type_Video
  - day_of_week_Monday
  - day_of_week_Saturday
  - day_of_week_Sunday
  - day_of_week_Thursday
  - day_of_week_Tuesday
  - day_of_week_Wednesday
  - content_topic_CampaignLaunch
  - content_topic_DonorImpact
  - content_topic_Education
  - content_topic_EventRecap
  - content_topic_Gratitude
  - content_topic_Health
  - content_topic_Reintegration
  - content_topic_SafehouseLife
  - sentiment_tone_Emotional
  - sentiment_ton

In [4]:
# Train/test split for sklearn deployment version
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    smp[categorical_features + binary_features + numeric_features],
    y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train_raw)}, Test size: {len(X_test_raw)}")

Train size: 649, Test size: 163


## Section 3 — Exploration

In [5]:
# Distribution of donation_referrals
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of target
axes[0, 0].hist(smp['donation_referrals'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[0, 0].set_title('Distribution of Donation Referrals')
axes[0, 0].set_xlabel('Donation Referrals')
axes[0, 0].set_ylabel('Frequency')

# Mean donation_referrals by platform
platform_means = smp.groupby('platform')['donation_referrals'].mean().sort_values(ascending=False)
axes[0, 1].bar(platform_means.index, platform_means.values, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Mean Donation Referrals by Platform')
axes[0, 1].set_xlabel('Platform')
axes[0, 1].set_ylabel('Mean Referrals')
axes[0, 1].tick_params(axis='x', rotation=45)

# Mean by post_type
pt_means = smp.groupby('post_type')['donation_referrals'].mean().sort_values(ascending=False)
axes[1, 0].barh(pt_means.index, pt_means.values, color='teal', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Mean Donation Referrals by Post Type')
axes[1, 0].set_xlabel('Mean Referrals')

# Scatter: caption_length vs referrals
axes[1, 1].scatter(smp['caption_length'], smp['donation_referrals'], alpha=0.3, color='coral')
axes[1, 1].set_title('Caption Length vs Donation Referrals')
axes[1, 1].set_xlabel('Caption Length')
axes[1, 1].set_ylabel('Donation Referrals')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/02_exploration.png', dpi=100, bbox_inches='tight')
plt.show()
print("Exploration plots generated")

Exploration plots generated


In [6]:
# Correlation heatmap of numeric features
numeric_cols = numeric_features + ['donation_referrals']
corr = smp[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/02_correlation.png', dpi=100, bbox_inches='tight')
plt.show()
print("Correlation heatmap generated")

Correlation heatmap generated


## Section 4 — Modeling (Explanatory OLS)

In [7]:
# Fit initial OLS model (explanatory — for coefficient interpretation)
model = sm.OLS(y, X_full).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:     donation_referrals   R-squared:                       0.333
Model:                            OLS   Adj. R-squared:                  0.298
Method:                 Least Squares   F-statistic:                     9.379
Date:                Mon, 06 Apr 2026   Prob (F-statistic):           8.15e-45
Time:                        15:15:37   Log-Likelihood:                -3782.4
No. Observations:                 812   AIC:                             7649.
Df Residuals:                     770   BIC:                             7846.
Df Model:                          41                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const           

In [8]:
# VIF check for multicollinearity
X_ols = X_full.copy()
vif_data = pd.DataFrame({
    'Feature': X_ols.columns[1:],  # Skip constant
    'VIF': [variance_inflation_factor(X_ols.values, i) for i in range(1, X_ols.shape[1])]
})
vif_data = vif_data.sort_values('VIF', ascending=False)
print("Variance Inflation Factors:")
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\nWARNING: {len(high_vif)} features with VIF > 10 detected.")
    print("Dropping high-VIF features and re-fitting...")
    drop_cols = high_vif['Feature'].tolist()
    X_ols = X_ols.drop(columns=drop_cols)
    model = sm.OLS(y, X_ols).fit()
    print("\nRe-fitted model summary:")
    print(model.summary())
    # Updated VIF
    vif_data2 = pd.DataFrame({
        'Feature': X_ols.columns[1:],
        'VIF': [variance_inflation_factor(X_ols.values, i) for i in range(1, X_ols.shape[1])]
    })
    print("\nUpdated VIF values:")
    print(vif_data2.sort_values('VIF', ascending=False).to_string(index=False))
else:
    print("\nAll VIF values <= 10. No multicollinearity issues detected.")

Variance Inflation Factors:
                     Feature       VIF
      follower_count_at_post 38.774824
           platform_LinkedIn 29.271149
            platform_YouTube 23.927683
             platform_TikTok 21.638061
            platform_Twitter 16.782785
           platform_WhatsApp  8.542475
          platform_Instagram  7.070043
       post_type_ImpactStory  5.912952
     features_resident_story  4.421437
            media_type_Video  2.798960
             media_type_Reel  2.613064
            media_type_Photo  2.557609
             media_type_Text  2.463710
          post_type_ThankYou  2.461108
     content_topic_Education  2.205021
 content_topic_SafehouseLife  2.163852
   content_topic_DonorImpact  2.091841
  sentiment_tone_Informative  2.018497
    post_type_EventPromotion  1.991449
      sentiment_tone_Hopeful  1.978433
    sentiment_tone_Emotional  1.945849
        content_topic_Health  1.923743
     sentiment_tone_Grateful  1.905700
 content_topic_Reintegration  1.8634


Updated VIF values:
                     Feature      VIF
       post_type_ImpactStory 5.889843
     features_resident_story 4.414809
            media_type_Photo 2.543591
            media_type_Video 2.524176
          post_type_ThankYou 2.439617
             media_type_Text 2.336750
     content_topic_Education 2.189459
 content_topic_SafehouseLife 2.139793
   content_topic_DonorImpact 2.078751
             media_type_Reel 2.035203
  sentiment_tone_Informative 2.015057
    post_type_EventPromotion 1.975949
      sentiment_tone_Hopeful 1.969951
    sentiment_tone_Emotional 1.938973
        content_topic_Health 1.918128
     sentiment_tone_Grateful 1.892495
       sentiment_tone_Urgent 1.842613
 content_topic_Reintegration 1.827011
     content_topic_Gratitude 1.825803
post_type_EducationalContent 1.825087
         day_of_week_Tuesday 1.787886
content_topic_CampaignLaunch 1.777447
              caption_length 1.767847
        day_of_week_Saturday 1.692504
          day_of_week_Monday 

### Coefficient Interpretation

The OLS model identifies which **controllable** post factors significantly predict donation referrals (p < 0.05). Positive coefficients indicate factors that *increase* referrals; negative coefficients indicate factors that *decrease* them, holding all other variables constant.

Key factors to examine:
- **Platform effects:** Which platforms generate more referrals
- **Post type:** Which content formats (FundraisingAppeal, EducationalContent, etc.) drive donations
- **Call to action:** Whether including a CTA significantly increases referrals
- **Resident story:** Whether featuring a resident's story increases referral likelihood
- **Boosted posts:** Whether paid promotion drives referrals
- **Content topic & sentiment:** Which topics/tones resonate with donors

## Section 5 — Evaluation

In [9]:
# Model fit statistics
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
y_pred_ols = model.fittedvalues
rmse = np.sqrt(mean_squared_error(y, y_pred_ols))
print(f"RMSE: {rmse:.4f}")
print(f"F-statistic: {model.fvalue:.2f}, p-value: {model.f_pvalue:.2e}")

R-squared: 0.3261
Adjusted R-squared: 0.2948
RMSE: 25.6478
F-statistic: 10.42, p-value: 9.39e-46


In [10]:
# Residual diagnostics
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residual plot (fitted vs residuals)
axes[0].scatter(fitted, residuals, alpha=0.3, color='coral')
axes[0].axhline(y=0, color='black', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted Values')

# Q-Q plot
sm.qqplot(residuals, line='45', ax=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

# Histogram of residuals
axes[2].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[2].set_title('Distribution of Residuals')
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/02_residuals.png', dpi=100, bbox_inches='tight')
plt.show()

In [11]:
# Shapiro-Wilk test for normality of residuals
if len(residuals) > 5000:
    stat, p_val = shapiro(residuals.sample(5000, random_state=42))
else:
    stat, p_val = shapiro(residuals)
print(f"Shapiro-Wilk Test: statistic={stat:.4f}, p-value={p_val:.4f}")
if p_val < 0.05:
    print("Residuals are NOT normally distributed (p < 0.05). This is common with count data.")
    print("Consider: count data often violates normality; OLS is still interpretable but CIs may be approximate.")
else:
    print("Residuals appear normally distributed (p >= 0.05).")

# Breusch-Pagan test for homoscedasticity
X_model = model.model.exog
bp_test = het_breuschpagan(residuals, X_model)
labels = ['LM Statistic', 'LM p-value', 'F-Statistic', 'F p-value']
print(f"\nBreusch-Pagan Test:")
for label, val in zip(labels, bp_test):
    print(f"  {label}: {val:.4f}")
if bp_test[1] < 0.05:
    print("Heteroscedasticity detected (p < 0.05). Standard errors may be biased.")
    print("Using HC3 robust standard errors for reliable inference.")
else:
    print("No evidence of heteroscedasticity (p >= 0.05).")

Shapiro-Wilk Test: statistic=0.6163, p-value=0.0000
Residuals are NOT normally distributed (p < 0.05). This is common with count data.
Consider: count data often violates normality; OLS is still interpretable but CIs may be approximate.

Breusch-Pagan Test:
  LM Statistic: 71.3674
  LM p-value: 0.0004
  F-Statistic: 2.0744
  F p-value: 0.0003
Heteroscedasticity detected (p < 0.05). Standard errors may be biased.
Using HC3 robust standard errors for reliable inference.


In [12]:
# Coefficient table with confidence intervals
coef_table = pd.DataFrame({
    'Coefficient': model.params,
    'Std Error': model.bse,
    'p-value': model.pvalues,
    'CI Lower': model.conf_int()[0],
    'CI Upper': model.conf_int()[1]
})
coef_table = coef_table.drop('const', errors='ignore')
coef_table = coef_table.sort_values('p-value')
print("Coefficient Table (sorted by significance):")
print(coef_table.to_string())

print(f"\nSignificant features (p < 0.05): {(coef_table['p-value'] < 0.05).sum()} of {len(coef_table)}")

Coefficient Table (sorted by significance):
                              Coefficient  Std Error       p-value   CI Lower   CI Upper
is_boosted                      14.071898   2.590728  7.479435e-08   8.986223  19.157573
features_resident_story         24.335854   4.810803  5.277355e-07  14.892105  33.779603
post_hour                        0.606421   0.148438  4.859247e-05   0.315033   0.897810
caption_length                   0.131210   0.042917  2.310408e-03   0.046962   0.215458
platform_WhatsApp               10.340752   3.521335  3.416592e-03   3.428267  17.253237
sentiment_tone_Informative      -9.534972   3.272541  3.675337e-03 -15.959068  -3.110876
sentiment_tone_Grateful         -9.653332   3.456282  5.351232e-03 -16.438117  -2.868547
sentiment_tone_Hopeful          -6.720156   3.340669  4.460573e-02 -13.277990  -0.162323
post_type_ImpactStory            9.650019   5.163580  6.201902e-02  -0.486242  19.786280
post_type_EducationalContent    -6.392234   3.582757  7.478786e-02

## Section 6 — Causal Analysis

In [13]:
# Significant coefficients interpretation
sig = coef_table[coef_table['p-value'] < 0.05].copy()

print("SIGNIFICANT FACTORS affecting donation referrals (p < 0.05):")
print("=" * 80)
if len(sig) == 0:
    print("  No features are individually significant at p < 0.05.")
    print("  This may indicate that donation referrals are driven by combinations")
    print("  of factors rather than individual predictors, or that the effect sizes")
    print("  are small relative to noise in the data.")
else:
    for idx, row in sig.iterrows():
        direction = "+" if row['Coefficient'] > 0 else ""
        print(f"  {idx}: {direction}{row['Coefficient']:.4f} (p={row['p-value']:.4f})")
        if row['Coefficient'] > 0:
            print(f"    -> A one-unit increase is associated with {row['Coefficient']:.2f} more donation referrals")
        else:
            print(f"    -> A one-unit increase is associated with {abs(row['Coefficient']):.2f} fewer donation referrals")
print()
print("Interpretation: Each coefficient represents the change in donation_referrals")
print("associated with a one-unit increase in the factor, holding other factors constant.")

SIGNIFICANT FACTORS affecting donation referrals (p < 0.05):
  is_boosted: +14.0719 (p=0.0000)
    -> A one-unit increase is associated with 14.07 more donation referrals
  features_resident_story: +24.3359 (p=0.0000)
    -> A one-unit increase is associated with 24.34 more donation referrals
  post_hour: +0.6064 (p=0.0000)
    -> A one-unit increase is associated with 0.61 more donation referrals
  caption_length: +0.1312 (p=0.0023)
    -> A one-unit increase is associated with 0.13 more donation referrals
  platform_WhatsApp: +10.3408 (p=0.0034)
    -> A one-unit increase is associated with 10.34 more donation referrals
  sentiment_tone_Informative: -9.5350 (p=0.0037)
    -> A one-unit increase is associated with 9.53 fewer donation referrals
  sentiment_tone_Grateful: -9.6533 (p=0.0054)
    -> A one-unit increase is associated with 9.65 fewer donation referrals
  sentiment_tone_Hopeful: -6.7202 (p=0.0446)
    -> A one-unit increase is associated with 6.72 fewer donation referrals

I

### Confounders and Limitations

1. **Seasonal Effects:** Donation behavior varies by time of year (holiday giving, end-of-year). The model does not account for temporal trends.
2. **Content Quality:** The emotional resonance and writing quality of captions is unmeasured. Two posts of the same type may differ dramatically in impact.
3. **Selection Bias in Boosted Posts:** Boosted posts may be selected *because* they performed well organically, making the boosting coefficient hard to interpret causally.
4. **Platform Algorithm Changes:** Social media algorithms change over time, affecting reach independent of post content.
5. **Audience Growth:** `follower_count_at_post` controls for audience size but not audience quality/engagement level.

### Actionable Recommendations

Based on the model results:
- **Focus on platforms and post types with positive significant coefficients** for donation campaigns
- **Include calls to action** when significant — direct CTAs may lower barriers to donate
- **Feature resident stories** if the coefficient is significantly positive — personal stories create emotional connection
- **Optimize hashtag count and caption length** based on their coefficient directions

## Section 7 - Deployment

**Deployment Architecture:** Pre-computed predictions written to PostgreSQL.

This model is deployed as an offline batch pipeline. The production workflow is:

1. **ETL:** `jobs/etl_social_media.py` reads social_media_posts from PostgreSQL, engineers features, and writes the `ml_social_media_features` table.
2. **Train:** `jobs/train_social_media.py` trains an OLS LinearRegression pipeline (sklearn Pipeline for consistency), saves the model as `artifacts/social_media.sav` along with `metadata.json` and `metrics.json`.
3. **Inference:** `jobs/run_inference_social_media.py` loads the trained model, predicts engagement for all posts, and writes results to the `social_media_predictions` table in PostgreSQL.

The .NET backend queries `social_media_predictions` via EF Core. The frontend fetches insights from `GET /api/predictions/social-media`.

**Model type:** Explanatory (OLS regression)

**Approach:** Explanatory -- which post factors drive donation-linked engagement. The sklearn Pipeline wraps the OLS model for inference consistency.

In [14]:
# For Flask API deployment: Train sklearn LinearRegression Pipeline
# (statsmodels OLS cannot be serialized with joblib for Flask)

cat_features_deploy = ['platform', 'post_type', 'media_type', 'day_of_week',
                        'content_topic', 'sentiment_tone']
num_features_deploy = ['has_call_to_action', 'features_resident_story', 'is_boosted',
                        'num_hashtags', 'caption_length', 'post_hour', 'follower_count_at_post']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features_deploy),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_features_deploy)
])

sklearn_pipeline = SkPipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

sklearn_pipeline.fit(X_train_raw, y_train)

# Evaluate sklearn pipeline on test set
y_pred_sk = sklearn_pipeline.predict(X_test_raw)
print(f"sklearn Pipeline Test R-squared: {r2_score(y_test, y_pred_sk):.4f}")
print(f"sklearn Pipeline Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_sk)):.4f}")

sklearn Pipeline Test R-squared: 0.3090
sklearn Pipeline Test RMSE: 27.3053


In [ ]:
# Save sklearn pipeline as .sav to artifacts/
joblib.dump(sklearn_pipeline, '../artifacts/social_media.sav')
print("Model saved to: ../artifacts/social_media.sav")

print("\nProduction scripts:")
print("  ETL:       jobs/etl_social_media.py")
print("  Train:     jobs/train_social_media.py")
print("  Inference: jobs/run_inference_social_media.py")
print("\nPredictions are pre-computed to PostgreSQL table: social_media_predictions")

In [16]:
# Expected input features for prediction
print("Expected input features (as DataFrame columns):")
print(f"  Categorical: {cat_features_deploy}")
print(f"  Numeric: {num_features_deploy}")
print()

# Example prediction
example = pd.DataFrame([{
    'platform': 'Facebook',
    'post_type': 'FundraisingAppeal',
    'media_type': 'Photo',
    'day_of_week': 'Monday',
    'content_topic': 'Fundraising',
    'sentiment_tone': 'Urgent',
    'has_call_to_action': 1,
    'features_resident_story': 1,
    'is_boosted': 0,
    'num_hashtags': 3,
    'caption_length': 150,
    'post_hour': 10,
    'follower_count_at_post': 2000
}])
pred = loaded.predict(example)
print(f"Example prediction (donation_referrals): {pred[0]:.2f}")

Expected input features (as DataFrame columns):
  Categorical: ['platform', 'post_type', 'media_type', 'day_of_week', 'content_topic', 'sentiment_tone']
  Numeric: ['has_call_to_action', 'features_resident_story', 'is_boosted', 'num_hashtags', 'caption_length', 'post_hour', 'follower_count_at_post']

Example prediction (donation_referrals): 44.07
